# Stage 4b. End-to-End Fine-tuning with Augmentation and Backbone Comparison

Baseline Stage 4 (full fusion, embedding ResNet18-CSA dibekukan) tidak mengalahkan Path A atau Path B saja, menandakan embedding ImageNet beku tidak menambah sinyal berguna untuk palm. Notebook ini mencoba fine-tuning end-to-end (badan backbone dibekukan, CSA dan proyeksi dilatih bersama fusion attention dan head lewat backpropagation penuh atas citra), berbeda dari percobaan serupa pada conjunctiva yang gagal (MAE memburuk dari 1.554 ke 1.794) karena tidak memakai augmentasi citra sama sekali. Di sini augmentasi ringan (flip, rotasi kecil, brightness/contrast) diaktifkan pada split train, dan dua backbone dibandingkan (ResNet18 vs MobileNetV3-Small) untuk menjawab keputusan arsitektur secara empiris pada data palm sendiri, bukan mewarisi pilihan conjunctiva begitu saja. Bobot kelas severity juga dicoba terpisah agar efek augmentasi dan efek bobot kelas tidak tercampur dalam satu angka.

## Environment Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
import torch

from configs import paths
from src.common import manifest as manifest_utils, train

output_dir = paths.outputs_dir("palm")
artifact_dir = paths.artifacts_dir("palm")

manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)
handcrafted = pd.read_csv(output_dir / "handcrafted_features.csv")

print("manifest", manifest.shape)
print("device", "cuda" if torch.cuda.is_available() else "cpu")

manifest (809, 15)
device cuda


## Configurations

Empat konfigurasi end-to-end dibandingkan, menyilangkan pilihan backbone dengan aktif tidaknya bobot kelas severity. Augmentasi selalu aktif pada semua konfigurasi end-to-end di notebook ini karena sudah menjadi prasyarat dasar sebelum perbandingan lain masuk akal (tanpa itu, hasil akan berulang seperti kegagalan conjunctiva).

In [2]:
configurations = {
    "e2e_resnet18": dict(backbone_name="resnet18", weight_severity_classes=False),
    "e2e_resnet18_weighted_severity": dict(backbone_name="resnet18", weight_severity_classes=True),
    "e2e_mobilenet_v3_small": dict(backbone_name="mobilenet_v3_small", weight_severity_classes=False),
    "e2e_mobilenet_v3_small_weighted_severity": dict(backbone_name="mobilenet_v3_small", weight_severity_classes=True),
}

## Train All Configurations

Batch size dan epoch dijaga moderat karena setiap epoch memuat dan menjalankan citra lewat backbone (bukan vektor fitur kecil seperti pada run_kfold biasa), jauh lebih mahal secara komputasi per epoch.

In [3]:
results = {}
for name, overrides in configurations.items():
    print("melatih", name)
    results[name] = train.run_kfold_end_to_end(
        manifest, handcrafted,
        n_splits=5, epochs=40, batch_size=32, learning_rate=1e-3,
        augment=True, **overrides,
    )
    fold_metrics = results[name]["fold_metrics"]
    print(
        name,
        "MAE", round(fold_metrics["mae"].mean(), 4),
        "accuracy", round(fold_metrics["accuracy"].mean(), 4),
        "severity_accuracy", round(fold_metrics["severity_accuracy"].mean(), 4),
    )

melatih e2e_resnet18


e2e_resnet18 MAE 0.8161 accuracy 0.6057 severity_accuracy 0.5921
melatih e2e_resnet18_weighted_severity


e2e_resnet18_weighted_severity MAE 0.7879 accuracy 0.6515 severity_accuracy 0.4772
melatih e2e_mobilenet_v3_small


e2e_mobilenet_v3_small MAE 0.7673 accuracy 0.6489 severity_accuracy 0.6539
melatih e2e_mobilenet_v3_small_weighted_severity


e2e_mobilenet_v3_small_weighted_severity MAE 0.7583 accuracy 0.6439 severity_accuracy 0.5117


## Verify CSA Actually Learned

Memastikan badan backbone tetap beku dan CSA benar-benar menerima gradien selama pelatihan, bukan sekadar klaim, mengikuti pola verifikasi yang sama dengan conjunctiva stage 5b.

In [4]:
from src.common import features

sample_model = results["e2e_resnet18"]["models"][0]
backbone = sample_model.embedding_backbone
print("early requires_grad (harus False)", next(backbone.early.parameters()).requires_grad)
print("late requires_grad (harus False)", next(backbone.late.parameters()).requires_grad)
print("csa requires_grad (harus True)", next(backbone.csa.parameters()).requires_grad)
print("projection requires_grad (harus True)", backbone.projection.weight.requires_grad)

fresh_backbone = features.EmbeddingBackbone(backbone_name="resnet18")
csa_weight_diff = (
    backbone.csa.channel_attention.mlp[0].weight.cpu() - fresh_backbone.csa.channel_attention.mlp[0].weight
).abs().mean().item()
print("rerata selisih bobot CSA terlatih vs inisialisasi acak baru (harus jauh dari nol)", round(csa_weight_diff, 6))

early requires_grad (harus False) False
late requires_grad (harus False) False
csa requires_grad (harus True) True
projection requires_grad (harus True) True
rerata selisih bobot CSA terlatih vs inisialisasi acak baru (harus jauh dari nol) 0.080899


## Compare Against Frozen-Embedding Baseline

Dibandingkan jujur dengan hasil Stage 4 (embedding dibekukan, CSA belum terlatih), apa pun hasilnya. Kappa severity ikut dihitung karena metrik akurasi saja tidak menunjukkan apakah kelas Moderate mulai terprediksi.

In [5]:
from sklearn.metrics import cohen_kappa_score

baseline_table = pd.read_csv(output_dir / "multitask_model_comparison.csv")

comparison_rows = list(baseline_table.to_dict(orient="records"))
for name, result in results.items():
    fold_metrics = result["fold_metrics"]
    oof = result["oof"]
    severity_valid = oof["severity_true"] >= 0
    kappa = (
        cohen_kappa_score(oof.loc[severity_valid, "severity_true"], oof.loc[severity_valid, "severity_pred"])
        if severity_valid.any() else float("nan")
    )
    comparison_rows.append({
        "configuration": name,
        "mae": fold_metrics["mae"].mean(),
        "rmse": fold_metrics["rmse"].mean(),
        "accuracy": fold_metrics["accuracy"].mean(),
        "severity_accuracy": fold_metrics["severity_accuracy"].mean(),
        "severity_kappa": kappa,
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table.round(4)

,configuration,mae,rmse,accuracy,severity_accuracy,severity_kappa
0,path_a_handcrafted,0.7475,0.9364,0.6402,0.6192,0.1155
1,path_b_deep,0.7419,0.9358,0.6427,0.6625,0.0315
2,full_fusion,0.7519,0.9475,0.6365,0.6057,0.1348
3,full_fusion_tuned,0.7510,0.9491,0.6304,0.6155,0.1864
4,full_fusion_weighted_severity,0.7505,0.9471,0.6415,0.4882,0.2153
5,path_b_deep_mobilenet,0.7476,0.9449,0.6477,0.6340,0.0459
6,full_fusion_mobilenet,0.7514,0.9454,0.6427,0.6217,0.1561
7,e2e_resnet18,0.8161,1.0299,0.6057,0.5921,0.1874
8,e2e_resnet18_weighted_severity,0.7879,1.0034,0.6515,0.4772,0.1814
9,e2e_mobilenet_v3_small,0.7673,0.9713,0.6489,0.6539,0.2127


## Save Results

Checkpoint per fold disimpan untuk setiap konfigurasi end-to-end agar dapat dipakai ulang pada stage evaluasi ablation tanpa melatih ulang.

In [6]:
comparison_table.to_csv(output_dir / "multitask_model_comparison_e2e.csv", index=False)

best_name = min(
    (name for name in results),
    key=lambda name: results[name]["fold_metrics"]["mae"].mean(),
)
results[best_name]["oof"].to_csv(output_dir / "multitask_oof_e2e_best.csv", index=False)
print("konfigurasi end-to-end terbaik berdasarkan MAE", best_name)

for name, result in results.items():
    for fold_index, fold_model in enumerate(result["models"]):
        checkpoint_path = artifact_dir / f"end_to_end_{name}_fold{fold_index}.pt"
        torch.save(fold_model.state_dict(), checkpoint_path)

print("saved end-to-end results and checkpoints to", output_dir, "and", artifact_dir)

konfigurasi end-to-end terbaik berdasarkan MAE e2e_mobilenet_v3_small_weighted_severity


saved end-to-end results and checkpoints to /home/praktikan/projects/Azril/hemavision/outputs/palm and /home/praktikan/projects/Azril/hemavision/artifacts/palm
